In [2]:
# pip install -r requirements.txt

In [18]:
import os
import numpy as np
import json
import sys
import time
from datetime import datetime
import spacy
from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from chromadb import Client
from chromadb.config import Settings


In [19]:
np.__version__

'1.26.4'

In [3]:
PERSISTENT_DIRECTORY = './chroma_db'
COLLECTION_NAME = 'local_collection'

# Initialize the console
console = Console()

# Set-up Chroma DB
settings = Settings(persist_directory=PERSISTENT_DIRECTORY, is_persistent=True)

# Initialize the Chroma DB client
client = Client(settings)

# Get or create the index/collection
collection = client.get_or_create_collection(COLLECTION_NAME) # creates at first time/ gets if already exists

In [4]:
text = "The quick brown fox jumps over the lazy dog. hahaha. It's the climb."

chunk_size = 3
overlap = 2
chunks = []
words = text.split()
for i in range(0, len(words), chunk_size - overlap): # defines the starting index for each chunk
    chunk = ' '.join(words[i:i + chunk_size])
    chunks.append(chunk)
chunks

['The quick brown',
 'quick brown fox',
 'brown fox jumps',
 'fox jumps over',
 'jumps over the',
 'over the lazy',
 'the lazy dog.',
 'lazy dog. hahaha.',
 "dog. hahaha. It's",
 "hahaha. It's the",
 "It's the climb.",
 'the climb.',
 'climb.']

In [5]:
# Load model
model = SentenceTransformer('all-MiniLM-L12-v2')

In [6]:
PROCESSED_FILES_PATH = 'processed_files.json'

if os.path.exists(PROCESSED_FILES_PATH):
    with open(PROCESSED_FILES_PATH, 'r') as f:
        json.load(f)

In [7]:
def get_embedding(text: str) -> list: # ok

    try:
        embedding = model.encode(text).tolist()
        return embedding
    except Exception as e:
        console.print(f'[red]Error: {e}[/red]')
        return None

test_emb = get_embedding(chunks[0])

In [17]:
process_file('docs/doc2.txt')

Added doc2.txt_0 to collection

In [15]:
PROCESSED_FILES_PATH = 'processed_files.json'

def load_processed_files():
    '''
    Load the processed files from the disk.

    Returns:
        dict: The processed files
    '''
    if os.path.exists(PROCESSED_FILES_PATH):
        with open(PROCESSED_FILES_PATH, 'r') as f:
            return json.load(f)
    else:
        return {}
    

def save_processed_files(processed_files):
    '''
    Save the processed files to disk.

    Args:
        processed_files (dict): The processed files
    '''
    with open(PROCESSED_FILES_PATH, 'w') as f:
        json.dump(processed_files, f)


def read_file(file_path: str) -> str:
    '''
    Read contents of file from docs folder.

    Args:
        file_path (str): The file path

    Returns:
        str: The content of the file
    '''
    try:
        with open(file_path, 'r') as f:
            return f.read()

    except Exception as e:
        console.print(f'[red]Error reading {file_path}: {e}[/red]')
        return ''


def split_text(text: str, chunk_size: int = 300, overlap: int = 100) -> list:
    '''
    Split the text into chunks.

    Args:
        text (str): The text to split
        chunk_size (int): The size of each chunk
        overlap (int): The overlap between chunks

    Returns:
        list: The list of chunks
    '''
    chunks = []
    words = text.split()
    for i in range(0, len(words), chunk_size - overlap): # defines the starting index for each chunk
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks


def process_file(file_path: str) -> None:
    '''
    Document ingestion.
    Process the file and add it to the index.

    Args:
        file_path (str): The file path
    '''
    file_name = os.path.basename(file_path)

    # Read the file
    text = read_file(file_path)
    if not text:
        return None
    
    # Split the text into chunks
    chunks = split_text(text)

    # Get the embeddings for the text
    vector_ids = []
    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        if not embedding:
            continue

        vector_id = f'{file_name}_{i}' # suffixes the file name with the chunk number
        vector_ids.append(vector_id)

        # Generate metadata
        metadata = {
            'file_name': file_name,
            'chunk_number': i,
            'timestamp': datetime.now().isoformat(),
            'preview': chunk[:30] # preview of the chunk
        }

        # Add the embedding to the collection
        # try:
            # print(embedding)
        collection.add(
            embeddings=[embedding],
            metadatas=[metadata],
            documents=[chunk],
            ids=[vector_id]

        )
        console.print(f'[green]Added {vector_id} to collection[/green]')
        # except Exception as e:
        #     console.print(f'[red]Error adding to collection: {e}[/red]')

    # processed = load_processed_files()
    # processed[file_name] = {
    #     'modified': os.path.getmtime(file_path), # gets last modified time
    #     'vector_ids': vector_ids,
    #     'name': file_name,
    # } # keeps track of the processed files
    # save_processed_files(processed)

    # console.print(f'[green]Processed {file_name} and uploaded to vector store.[/green]')


def delete_vectors(file_name: str):

    processed = load_processed_files()
    file_data = processed.get(file_name)
    if not file_data:
        console.print(f'[red]File {file_name} not found[/red]')
        return
    
    try:
        collection.delete(where={'file_name': file_name}) # from metadata
        console.print(f'[green]Deleted vectors for {file_name}[/green]')
    except Exception as e:
        console.print(f'[red]Error deleting vectors: {e}[/red]')


def list_files():
    ''''
    List the files that have been processed and stored in the local folder.
    '''
    folder_path = 'docs'
    if not os.path.exists(folder_path):
        console.print(f'[red]Folder {folder_path} not found[/red]')
        os.mkdir(folder_path)
        console.print(f'[green]Created folder {folder_path}[/green]')

    processed = load_processed_files()

    try:
        current_files = os.listdir(folder_path)
        for file_name in current_files:
            file_path = os.path.join(folder_path, file_name)
            if os.path.isfile(file_path):
                processed_data = processed.get(file_name)
                if processed_data:
                    console.print(f'[green]{file_name}[/green]')
                else:
                    console.print(f'[red]{file_name}[/red]')
    except Exception as e:
        console.print(f'[red]Error: {e}[/red]')


def update_files():
    '''
    Update the files in the docs folder.
    '''
    folder_path = "docs"
    if not os.path.exists(folder_path):
        console.print(f"[red]Folder {folder_path} not found[/red]")
        return

    processed = load_processed_files()

    for file_name in os.listdir(folder_path):
        print(file_name)
        file_path = os.path.join(folder_path, file_name)

        if os.path.isfile(file_path):
            processed_data = processed.get(file_name, {})
            modified_time = os.path.getmtime(file_path)

            if processed_data and modified_time > processed_data.get("modified", 0):
                console.print(f"[yellow]Updating {file_name}[/yellow]")
                delete_vectors(file_name)

            console.print(f"[yellow]Processing {file_name}[/yellow]")
            process_file(file_path)

In [3]:
os.path.exists('docs')

True

In [4]:
os.listdir('docs')

['doc5.txt', 'doc4.txt', 'doc1.txt', 'doc3.txt', 'doc2.txt']

In [10]:
with open('docs/doc1.txt', 'r') as f:
    print(f.read())

Car Loan Eligibility Criteria
To qualify for a car loan, applicants must meet the following criteria:

1. Age: Must be between 21 and 65 years old at loan maturity.

2. Employment: Should be salaried or self-employed with a stable income.

3. Credit Score: A minimum credit score of 650 is required.

4. Income: Monthly income should meet or exceed the lender's minimum requirement.

5. Residency: Must be a resident or citizen of the country where the loan is applied.


In [11]:
os.path.basename('docs/doc1.txt')

'doc1.txt'

In [13]:
os.path.getmtime('docs/doc1.txt')

1742099250.607932

### new functions

In [ ]:
# Load model



In [16]:
text = 'the quick brown fox jumps over the lazy dog'

embedding = model.encode(text).tolist()
print(embedding)

[0.0072746980004012585, 0.03941640257835388, -0.03635295480489731, 0.03405699133872986, 0.07380785793066025, -0.0037293133791536093, 0.0243414044380188, -0.016939276829361916, -0.019710756838321686, -0.047845207154750824, -0.08388354629278183, 0.024791955947875977, -0.02630561590194702, -0.025797182694077492, -0.04281601309776306, -0.005831190850585699, -0.050882771611213684, -0.01050784531980753, -0.0004160422249697149, -0.034269314259290695, -0.06622568517923355, -0.11466453969478607, 0.015677427873015404, -0.07399358600378036, -0.14477530121803284, -0.017294414341449738, 0.009208200499415398, 0.04245380312204361, 0.01216132566332817, -0.07140979915857315, 0.017273763194680214, -0.08868570625782013, -0.0018613425781950355, 0.03333120048046112, -0.05717632174491882, -0.020472796633839607, 0.034442879259586334, -0.0495133139193058, 0.01582495868206024, 0.026905594393610954, -0.034575194120407104, -0.07433867454528809, -0.009409371763467789, 0.015217105858027935, -0.09809861332178116, -

In [24]:
chunked_text = nlp(text)

print([(w.text, w.pos_) for w in chunked_text])

[('the', 'DET'), ('quick', 'ADJ'), ('brown', 'ADJ'), ('fox', 'NOUN'), ('jumps', 'VERB'), ('over', 'ADP'), ('the', 'DET'), ('lazy', 'ADJ'), ('dog', 'NOUN')]


In [37]:
spacy.load('en_core_web_sm')
import en_core_web_sm
nlp = en_core_web_sm.load()

def chunk_text(text: str, chunk_size=30, overlap=5) -> list:
    
    doc = nlp(text)
    chunks = []
    start = 0
    while start < len(doc):
        end = min(start + chunk_size, len(doc))
        chunks.append(doc[start:end].text)
        start += chunk_size - overlap
    return chunks

In [ ]:
model = SentenceTransformer('all-MiniLM-L12-v2')

def get_embedding(chunk):

    embedding = model.encode(chunk).tolist()
    return embedding

In [46]:
def add_vector():

    collection.add(
        embeddings=[chunk_embed],
        metadatas=[chunk_metadata],
        documents=[chunk],
        ids=[chunk_id],
        )

In [ ]:
def update_files():

    doc_folder = 'docs'

    if os.path.exists(doc_folder):
        for file in os.listdir(doc_folder): # iterate through files in folder
            
            file_name = os.path.basename(file)

            file_timestamp = os.path.getmtime(file)

            with open(os.path.join(doc_folder, file), 'r') as f:
                file_content = f.read()
                file_content_chunks = chunk_text(file_content)

            for id_, chunk in enumerate(file_content_chunks):

                chunk_embed = get_embedding(chunk)

                chunk_id = f'{file_name}_{id_}'

                chunk_metadata = {'file_name': file_name,
                                  'chunk_number': id_,
                                  'timestamp': file_timestamp,
                                  'preview': chunk[:30]
                                  }
                
                add_vector()


            tracker_json_path = 'processed_files.json'
            if os.path.exists(tracker_json_path)
                ...
            else:
                tracker_json = {}
                tracker_json['file_name'] = file_name
                tracker_json['timestamp'] = os.path.getmtime(file)

                tracker_json['ids'] = # insert ids from embed_chunks here




            